# Baseline models

A **baseline** is a deliberately simple model that gives us a score to beat. Without one, a
number like "92% accuracy" means little. We start with three, in increasing order of complexity:

1. Always predict "no stress" (the majority class).
2. A one-line rule using only current inflation (the persistence rule).
3. Logistic regression with 6 features.

Every experiment goes into `results/model_log.csv`, including the ones that perform badly.
Run this notebook once per experiment: each run adds new rows to the log.

In [1]:
import sys; sys.path.append('../src')
import pandas as pd, numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from config import *
from utils import load_panel, time_split, evaluate, log_result

## Load the data with the shared split

We train on 2010-2021 and test on 2022-2024. The split follows time rather than being random,
because in real use we can only learn from the past. Everyone uses `time_split`, so all models
are scored on the same 404 test rows.

In [2]:
df = load_panel()
FEATURES = ['brent_yoy_pct', 'inflation_pct', 'gdp_growth_pct', 'import_burden_pct_gdp', 'oil_intensity', 'fx_change_pct']
X_tr, X_te, y_tr, y_te = time_split(df, FEATURES)
print('Training rows:', X_tr.shape[0], '| stress rate:', round(y_tr.mean(), 3))
print('Test rows:    ', X_te.shape[0], '| stress rate:', round(y_te.mean(), 3), '| stress cases:', int(y_te.sum()))

results = []  # collected for the comparison table at the end

def record(name, y_pred, metrics):
    """Keep a model's scores and its confusion matrix (counts of right and wrong calls)."""
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    results.append({'model': name, **metrics, 'true_alarms': tp, 'false_alarms': fp, 'missed': fn})

Training rows: 1701 | stress rate: 0.12
Test rows:     404 | stress rate: 0.144 | stress cases: 58


## Baseline 0: always predict "no stress"

**What it does:** it ignores the features and predicts the most common outcome.

**Why we run it:** it shows what accuracy is worth on imbalanced data. **Precision** is the share
of alarms that were correct. **Recall** is the share of real stress cases that were caught.
**F1** combines the two into a single score, and it is low if either one is low.

In [3]:
majority = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)
pred = majority.predict(X_te)
metrics = evaluate(y_te, pred)
log_result('majority_baseline', {'strategy': 'most_frequent'}, metrics,
           'Starting point: ignores all features. Shows accuracy alone is misleading on imbalanced data.')
record('Always "no stress"', pred, metrics)
metrics

{'accuracy': 0.8564, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

**Takeaway:** 85.6% accuracy while catching no stress cases at all. This is why we do not judge
models on accuracy.

## Baseline 1: the persistence rule

**What it does:** it flags a country if its inflation is above 10% this year. There is nothing to
train.

**Why we run it:** open issue 1 says current inflation may explain most of the predictive power.
This rule is the fairest benchmark: a model only adds value if it beats a one-line rule.

**ROC-AUC** measures how well a score ranks risky countries above safe ones, where 1.0 is perfect
and 0.5 is random. Here the score is simply the inflation rate. A blank inflation value gets the
training median for ranking and is not flagged.

In [4]:
pred = (X_te['inflation_pct'] > INFLATION_STRESS_THRESHOLD).astype(int)
score = X_te['inflation_pct'].fillna(X_tr['inflation_pct'].median())
metrics = evaluate(y_te, pred, score)
log_result('persistence_rule', {'rule': 'inflation_pct > 10'}, metrics,
           'One-line benchmark for the persistence effect (open issue 1). No training. A model must beat this to add value.')
record('Persistence rule', pred, metrics)
persistence_pred = pred
metrics

{'accuracy': 0.8837,
 'precision': 0.5591,
 'recall': 0.8966,
 'f1': 0.6887,
 'roc_auc': 0.9511}

**Takeaway:** this one-line rule catches 90% of stress cases with a ROC-AUC of 0.951, but about
44% of its alarms are false. It is a strong benchmark to beat.

## Baseline 2: logistic regression

**Main idea:** logistic regression gives each feature a weight, adds them up, and converts the
total into a probability between 0 and 1. By default it flags a country when that probability is
above 50%.

**Preprocessing:** empty values are **imputed** (filled in) with the median, and features are
**scaled** to a common range so that large numbers don't dominate. Both steps sit inside a
**Pipeline**, which learns them from training rows only, so no test information leaks in.

In [5]:
logreg = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                       LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
logreg.fit(X_tr, y_tr)
pred = logreg.predict(X_te)
metrics = evaluate(y_te, pred, logreg.predict_proba(X_te)[:, 1])
log_result('logreg_v1', {'C': 1.0, 'imputer': 'median', 'scaler': 'standard', 'n_features': 6}, metrics,
           'First real model: 6 features, default settings. Fewer false alarms than the persistence rule but similar ranking.')
record('Logistic regression (6 features)', pred, metrics)
logreg_pred = pred

weights = pd.Series(logreg[-1].coef_[0], index=FEATURES).sort_values(key=abs, ascending=False).round(3)
display(weights.rename('weight (scaled features)'))
metrics

inflation_pct            4.159
brent_yoy_pct            0.447
import_burden_pct_gdp   -0.221
oil_intensity            0.158
gdp_growth_pct           0.072
fx_change_pct            0.025
Name: weight (scaled features), dtype: float64

{'accuracy': 0.9208,
 'precision': 0.6857,
 'recall': 0.8276,
 'f1': 0.75,
 'roc_auc': 0.9481}

**Takeaway:** the highest F1 so far (0.750), but current inflation carries by far the largest
weight (4.16, against 0.45 for the next largest, Brent). The model is mostly a refined version of
the persistence rule.

## Comparison

All three models on the same test rows. **False alarms** are countries flagged that did not go
into stress. **Missed** cases are stress years that were not flagged.

In [6]:
comparison = pd.DataFrame(results).set_index('model')
comparison.to_csv(TABLES / 'baseline_results.csv')
comparison

,accuracy,precision,recall,f1,true_alarms,false_alarms,missed,roc_auc
model,,,,,,,,
"Always ""no stress""",0.8564,0.0000,0.0000,0.0000,0,0,58,NaN
Persistence rule,0.8837,0.5591,0.8966,0.6887,52,41,6,0.9511
Logistic regression (6 features),0.9208,0.6857,0.8276,0.7500,48,22,10,0.9481


**Takeaway:** logistic regression beats the persistence rule on F1 because it raises about half as many
false alarms (22 vs 41), though it catches slightly fewer real cases (48 vs 52).
Their ROC-AUC scores are nearly identical (0.948 vs 0.951). With only 58 test stress cases, one
case moves recall by 1.7 percentage points, so small gaps should not be over-read.

## Can the models spot stress that is new?

The persistence rule cannot catch **new-onset** stress by construction, since those countries are
at or below 10% this year. Does logistic regression do any better?

In [7]:
test_rows = df[df['year'] >= TEST_START_YEAR].copy()
test_rows['logreg_flag'] = logreg_pred
test_rows['rule_flag'] = persistence_pred.values
new_onset = test_rows[(test_rows['stress_next'] == 1) & (test_rows['inflation_pct'] <= INFLATION_STRESS_THRESHOLD)]
new_onset = new_onset[['iso3', 'country_name', 'year', 'inflation_pct', 'inflation_next', 'logreg_flag', 'rule_flag']].round(1)
new_onset.to_csv(TABLES / 'new_onset_test_cases.csv', index=False)
print('New-onset stress cases in the test set:', len(new_onset))
print('Caught by logistic regression:', int(new_onset['logreg_flag'].sum()))
new_onset

New-onset stress cases in the test set: 6
Caught by logistic regression: 0


,iso3,country_name,year,inflation_pct,inflation_next,logreg_flag,rule_flag
184,BGD,Bangladesh,2023,9.9,10.5,0.0,0
273,BOL,Bolivia,2024,5.1,19.5,0.0,0
1026,KAZ,Kazakhstan,2024,8.7,11.4,0.0,0
1665,PSE,West Bank and Gaza,2023,5.9,53.7,0.0,0
1907,TCD,Chad,2022,5.8,10.8,0.0,0
2005,UKR,Ukraine,2024,6.5,12.7,0.0,0


**Takeaway:** the test set has only 6 new-onset cases and neither baseline catches any of them.
Every stress case logistic regression caught was already above 10% inflation. So the baselines
do not yet answer the business question of flagging new trouble a year in advance. Phase 2 needs
to test whether oil features change this, and the separate test on 2021 rows gives it a much
larger sample of new cases (36 of the 50 stress cases there are new-onset).

**Next (Phase 2):** missing-value strategies, feature engineering, clustering, macro-only vs
macro + oil, and then other algorithms.